# Orquestador de entrenamiento y predicción

Este cuaderno ejecuta, en orden, los scripts `src/train.py` y `src/predict.py` utilizando el mismo intérprete de Python del kernel.

Se conservan los scripts como componentes independientes y se muestran en la salida del cuaderno sus mensajes de consola, métricas, tiempos y errores. Si el entrenamiento falla, la predicción no se ejecuta.

## 1. Identificar la raíz del proyecto

Abre este cuaderno desde la raíz del repositorio o desde una subcarpeta del proyecto. El código busca automáticamente la carpeta que contiene `src/train.py` y `src/predict.py`.

In [1]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src" / "train.py").is_file() and (candidate / "src" / "predict.py").is_file():
            return candidate
    raise FileNotFoundError(
        "No se encontró la raíz del proyecto. Comprueba que el cuaderno esté dentro de "
        "DataScientist_wizeline y que exista src/train.py y src/predict.py."
    )

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
TRAIN_SCRIPT = PROJECT_ROOT / "src" / "train.py"
PREDICT_SCRIPT = PROJECT_ROOT / "src" / "predict.py"
PREDICTIONS_PATH = PROJECT_ROOT / "outputs" / "blind_test_predictions.csv"

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Intérprete de Python: {sys.executable}")
print(f"Script de entrenamiento: {TRAIN_SCRIPT}")
print(f"Script de predicción: {PREDICT_SCRIPT}")


Raíz del proyecto: C:\Users\angry\source\repos\repos_otros\6.Wizeline_test\DataScientist_wizeline
Intérprete de Python: c:\Users\angry\source\repos\repos_otros\6.Wizeline_test\DataScientist_wizeline\.venv\Scripts\python.exe
Script de entrenamiento: C:\Users\angry\source\repos\repos_otros\6.Wizeline_test\DataScientist_wizeline\src\train.py
Script de predicción: C:\Users\angry\source\repos\repos_otros\6.Wizeline_test\DataScientist_wizeline\src\predict.py


## 2. Función para ejecutar scripts y mostrar sus mensajes

La salida estándar y los errores se muestran en tiempo real en la celda. El proceso utiliza `sys.executable`, por lo que usa el Python asociado al kernel del cuaderno (selecciona el kernel de `.venv` antes de ejecutar).

Para evitar problemas de acentuación en Windows, los procesos hijos fuerzan UTF-8 mediante `PYTHONUTF8` y `PYTHONIOENCODING`, y el orquestador decodifica la salida explícitamente con `encoding="utf-8"`.


In [2]:
import subprocess
import os

def run_script(script_path: Path, stage_name: str) -> None:
    if not script_path.is_file():
        raise FileNotFoundError(f"No se encontró el script: {script_path}")

    print("=" * 72, flush=True)
    print(f"INICIO: {stage_name}", flush=True)
    print(f"Script: {script_path}", flush=True)
    print("=" * 72, flush=True)

    # Fuerza UTF-8 en los procesos Python hijos, independientemente
    # de la página de códigos configurada en Windows.
    process_env = os.environ.copy()
    process_env["PYTHONUTF8"] = "1"
    process_env["PYTHONIOENCODING"] = "utf-8"

    process = subprocess.Popen(
        [sys.executable, "-u", str(script_path)],
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
        env=process_env,
    )

    assert process.stdout is not None
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
    finally:
        process.stdout.close()

    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, process.args)

    print(f"\\nFIN: {stage_name} (código de salida 0)", flush=True)


## 3. Ejecutar entrenamiento y después predicción

Ejecuta esta celda para lanzar ambos procesos en secuencia. La predicción solo comienza si el entrenamiento termina correctamente.

In [3]:
run_script(TRAIN_SCRIPT, "ENTRENAMIENTO Y EVALUACIÓN")
run_script(PREDICT_SCRIPT, "PREDICCIÓN SOBRE EL CONJUNTO CIEGO")

print("\nFlujo completo finalizado correctamente.")
print(f"Archivo de predicciones esperado: {PREDICTIONS_PATH}")


INICIO: ENTRENAMIENTO Y EVALUACIÓN
Script: C:\Users\angry\source\repos\repos_otros\6.Wizeline_test\DataScientist_wizeline\src\train.py
ENTRENAMIENTO Y EVALUACIÓN DE REGRESIÓN
Filas totales: 800 | desarrollo: 640 | test reservado: 160
Predictores originales: 20 | CV: 5 folds
Correlación absoluta permitida: [0.05, 0.95]
Buscando configuración con menor RMSE medio de validación cruzada...
Procesadores lógicos disponibles: 12
Trabajos de entrenamiento concurrentes: 11
Fitting 5 folds for each of 128 candidates, totalling 640 fits
Tiempo de búsqueda CV: 12.83 segundos (0.21 minutos)

Modelo aceptado y guardado: C:\Users\angry\source\repos\repos_otros\6.Wizeline_test\DataScientist_wizeline\models\best_model.joblib

--- Resultado ---
Modelo seleccionado: XGBRegressor
Mejor RMSE medio CV: 2.205
RMSE desarrollo (80 %): 1.433
RMSE test reservado (20 %): 1.955
MAE test: 1.612 | R² test: 0.830
RMSE entrenamiento CV: 1.438
RMSE validación CV: 2.205
Brecha absoluta CV: 0.766
Brecha absoluta entrenam

## 4. Comprobar el archivo de predicciones

Esta celda comprueba la existencia del CSV, la columna `target_pred`, el número de filas y la validez numérica de las predicciones. No vuelve a entrenar ni modifica el archivo.

In [4]:
import numpy as np
import pandas as pd

if not PREDICTIONS_PATH.is_file():
    raise FileNotFoundError(f"No se generó el archivo esperado: {PREDICTIONS_PATH}")

predictions_df = pd.read_csv(PREDICTIONS_PATH)

if "target_pred" not in predictions_df.columns:
    raise ValueError("El archivo no contiene la columna requerida 'target_pred'.")

predicted_values = pd.to_numeric(predictions_df["target_pred"], errors="coerce")
if predicted_values.isna().any() or not np.isfinite(predicted_values.to_numpy()).all():
    raise ValueError("'target_pred' contiene valores faltantes o no finitos.")

print(f"Archivo: {PREDICTIONS_PATH}")
print(f"Filas: {len(predictions_df)}")
print(f"Columnas totales: {len(predictions_df.columns)}")
print(f"Valores faltantes en target_pred: {predicted_values.isna().sum()}")
print(f"Mínimo de target_pred: {predicted_values.min():.3f}")
print(f"Máximo de target_pred: {predicted_values.max():.3f}")
print(f"Media de target_pred: {predicted_values.mean():.3f}")

display(predictions_df.head())


Archivo: C:\Users\angry\source\repos\repos_otros\6.Wizeline_test\DataScientist_wizeline\outputs\blind_test_predictions.csv
Filas: 200
Columnas totales: 21
Valores faltantes en target_pred: 0
Mínimo de target_pred: 1.973
Máximo de target_pred: 25.188
Media de target_pred: 14.176


,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,target_pred
0,676.867615,32.518822,254.825875,502.268510,609.469688,497.624266,105.246239,269.045539,150.177005,312.649860,...,0.237996,660.030637,147.059794,40.232132,464.424834,121.147466,68.284243,26.969870,314.461582,12.299208
1,628.695228,426.163933,347.070280,431.106903,915.527507,301.699534,1.666992,306.733041,104.234252,63.242070,...,6.608084,600.994184,43.619815,48.153926,457.256565,49.163652,85.511662,33.500538,819.537877,12.642783
2,131.765943,323.839669,245.399775,181.814398,710.179159,59.117377,312.622788,687.965027,109.803179,381.169500,...,1.822370,736.306092,138.759029,36.915389,436.174065,10.037994,62.631938,6.211169,341.361374,14.656399
3,160.970195,489.712029,70.482159,309.486269,888.030604,412.655666,216.124989,47.415477,104.139145,326.462385,...,1.686895,485.144327,143.668518,27.168148,309.715497,149.661493,66.415878,15.001753,539.087409,12.173369
4,419.907137,216.625219,487.887860,253.704462,323.226862,65.744463,271.811469,527.726782,129.805782,168.429679,...,0.948507,365.946758,72.337904,36.232169,302.772338,186.944884,106.514846,3.443809,364.341969,18.646908


## Orden recomendado de ejecución

1. Selecciona como kernel el entorno `.venv` del proyecto.
2. Ejecuta las celdas de arriba hacia abajo.
3. Para repetir todo el flujo, vuelve a ejecutar la celda de la sección 3; luego ejecuta la sección 4 para revisar el CSV recién generado.